### Hash Maps
#### The Core Insight

> **A hash map trades memory for the ability to answer "have I seen this before?" and "where did I see it?" in $O(1)$ instead of $O(n)$.**

The redundant work in brute force is **scanning the array repeatedly to find something you've already passed**. A hash map remembers what you've seen the moment you see it — then answers lookup questions **instantly**.

#### The Mental Model

**Brute force — scan every time**:
> "Does 9 - 4 = 5 exist in the array?"
```shell
[2, 4, 7, 5, 1]  →  scan left to right... found at index 3
                     O(n) per lookup × n elements = O(n²)
```

**Hash map — remember as you go**:
```shell
seen = {}
Step 1: see 2  → seen = {2: 0}          "store what I've seen"
Step 2: see 4  → seen = {2:0, 4:1}
Step 3: see 7  → seen = {2:0, 4:1, 7:2}
Step 4: see 5  → need = 9 - 5 = 4
                 4 in seen? YES → O(1)   "answer instantly"
```
One pass. $O(n)$ total.

---

#### What Hash Maps and Sets Actually Give You
```shell
HASH MAP  (dict)               HASH SET  (set)
────────────────────────────   ──────────────────────────────
Key → Value                    Value only (key with no value)
"where did I see X?"           "have I seen X?"
"how many times did I see X?"  "is X in this collection?"
"what is the index of X?"      membership test only

dict: {value: index}           set: {value}
dict: {value: count}
dict: {value: [list of data]}

Lookup:   O(1) average         O(1) average
Insert:   O(1) average         O(1) average
Delete:   O(1) average         O(1) average
vs array: O(n) for all three
```

---

#### The Three Core Jobs
**Job 1 — Existence Check (set or dict)**
```py
# "Have I seen this element before?"
# Brute force: scan array O(n) per check
# Hash set:    O(1) per check

# Classic: Two Sum
def two_sum(nums: list[int], target: int) -> list[int]:
    seen = {}                               # value → index

    for i, num in enumerate(nums):
        complement = target - num
        if complement in seen:              # O(1) lookup; seen only holds PAST elements
            return [seen[complement], i]    # so num never pairs with itself ✅
        seen[num] = i                       # O(1)

    return []
```
**Job 2 — Frequency Count (dict / Counter)**
```py
# "How many times have I seen each element?"
# This turns frequency problems from O(n²) → O(n)

from collections import Counter

# Valid Anagram
def is_anagram(s: str, t: str) -> bool:
    return Counter(s) == Counter(t)
    # Counter builds freq map in O(n), comparison in O(k)
    # k = unique chars — at most 26 for lowercase English

# Manually:
def is_anagram(s: str, t: str) -> bool:
    freq = {}
    for ch in s:
        freq[ch] = freq.get(ch, 0) + 1   # count up
    for ch in t:
        freq[ch] = freq.get(ch, 0) - 1   # count down
    return all(v == 0 for v in freq.values())
```
**Job 3 — Index / Position Memory (dict)**
```py
# "WHERE did I see this, not just IF I saw it?"
# The hash map stores the index alongside the value

# Longest Substring Without Repeating Characters
def length_of_longest_substring(s: str) -> int:
    last_seen = {}    # char → last index where it appeared
    left   = 0
    result = 0

    for right, ch in enumerate(s):
        if ch in last_seen and last_seen[ch] >= left:
            left = last_seen[ch] + 1    # jump left past the repeat
        last_seen[ch] = right           # remember latest position
        result = max(result, right - left + 1)

    return result

# The hash map here replaces a full rescan to find
# "where did this character last appear?" — O(1) not O(n)
```

---

#### Core Advantage Over Arrays
```shell
OPERATION         ARRAY        HASH MAP / SET
─────────────────────────────────────────────
Search for value  O(n)         O(1)  ← the entire advantage
Insert            O(1) amort.  O(1)
Delete by value   O(n)         O(1)
Delete by index   O(n)         N/A
Preserve order    ✅           ❌  (dict preserves insert order,
                                    but not sorted order)
Duplicates        ✅           ❌  (set removes them)
Memory            O(n)         O(n)  (but more overhead per element)
```

---

#### The Signal — When to Reach for It
```shell
"Does the problem involve..."

  Finding a complement / pair          → dict {value: index}
  Detecting a duplicate                → set
  Counting frequencies                 → dict or Counter
  Grouping elements by some property   → dict {key: [list]}
  Checking if two structures match     → Counter == Counter
  Remembering the last position of X   → dict {value: last_index}
  Finding the first non-repeating X    → dict + second pass

The pattern in one question:
  "Do I need to find, count, or remember something
   I've already seen?"

  YES → hash map or set
```

---

#### The Classic Mistakes
```py
# ── Mistake 1: checking the array instead of the set ──────────
# ❌ O(n) lookup inside O(n) loop = O(n²)
for num in nums:
    if (target - num) in nums:       # 'in' on a list = O(n)!
        ...

# ✅ O(1) lookup
seen = set(nums)
for num in nums:
    if (target - num) in seen:       # 'in' on a set = O(1)
        ...


# ── Mistake 2: forgetting the element can't pair with itself ──
# Two Sum: [3, 5] target=6 — should 3+3 count if only one 3?
seen = {}
for i, num in enumerate(nums):
    comp = target - num
    if comp in seen:                 # seen only holds PAST elements
        return [seen[comp], i]       # so num never pairs with itself ✅
    seen[num] = i


# ── Mistake 3: using a list when a set is enough ──────────────
# ❌ wastes memory, slower lookup
seen = []
if x in seen: ...      # O(n)

# ✅
seen = set()
if x in seen: ...      # O(1)


# ── Mistake 4: counting with dict instead of Counter ──────────
# ❌ verbose
freq = {}
for ch in s:
    if ch not in freq:
        freq[ch] = 0
    freq[ch] += 1

# ✅ idiomatic
from collections import Counter
freq = Counter(s)
# also gives: freq.most_common(3), freq.elements()
```

---

#### The One-Sentence Core
> A hash map is a memory of the past — it lets you answer questions about **previously seen elements** in $O(1)$, turning a second scan of the array into a single dictionary lookup.

---

#### Pattern Side by Side
```shell
Sliding Window                    Hash Map / Set
──────────────────────────────    ──────────────────────────────
Contiguous subarray/substring     Any elements, any positions
Maintains a window of elements    Maintains a memory of elements
Shrinks/grows by left/right       Grows as you iterate
Eliminates recomputation          Eliminates rescanning
"sum/max/min of subarray"         "pair/duplicate/frequency"

Often combined:
  Sliding window tracks the BOUNDARY
  Hash map tracks what's INSIDE the window
  → "longest substring with at most K distinct characters"
```

---

### Hash Map Breakdown
**What problem does a Hash Map solve that a plain array or list cannot?**
- So, first off, hash map has constant $O(1)$ lookup, insertion, deletion time, which are $O(n)$ for arrays. This is the main benefit. It stores key-value pairs, so it easier to access items knowing the name of the key, in arrays you only have indexing to access items, not that convenient. Both are mutable data types.

Good start, but let's pin down one part before moving on. You said array lookup/insertion/deletion are $O(n)$ — is that true for **all** array operations, or just some?

Take `arr = [10, 20, 30, 40]`.

- What's the time complexity of `arr[2]`?
- Now what's the time complexity of answering "is 30 in this array?" or "what index is 30 at?"

**Are those the same operation, or different?**
- $O(n$) is true for some array operations, but yes, for appending to the end of the array and popping from the end is indeed $O(1)$, it adds/takes items from the end, no scanning the whole array. But deletion and insertion in the middle or the beginning will take reindexing for the rest of items, so it will surely cost $O(n)$. Also lookup by index is indeed $O(1)$, that is true.

1. `arr[2]` will be $O(1)$ lookup time, like for hash map by key.
2. `in` membership check for lists will cost $O(n)$, it will **scan the entire array** one by one, as well as finding index for a specified element. For dicts and sets `in` membership check costs $O(1)$, constant time.
So, accessing array's element by index is $O(1)$ while `in` membership and finding an element's index costs scanning the whole array with $O(n)$.

Correct on all counts — that's the precise breakdown.

Array operations split into **two buckets**:
1. **position-based** (index access, end append/pop) are $O(1)$;
2. **content-based** (search, membership, arbitrary insert/delete) require touching potentially every element, so $O(n)$.

> **That's the problem hash maps solve**: they turn the content-based operations — "is this key here?", "give me the value for this key" — into $O(1)$, which arrays fundamentally cannot do.

Now the harder question: **how**? An array gets $O(1)$ index access because `arr[i]` is just `base_address + i * element_size` — pure math, no searching. 

**A hash map has *no numeric index*, just a key like `"apple"`. What do you think has to happen internally so that looking up `"apple"` doesn't require scanning?**
- A hash map uses a **hash function** to compute a **hash key** when we store a value under its key. We give a key-value pair, where the key must be **hashable object** in order to be used with the hash function. Then, knowing the hash value the hash map computes a slot in the array, where the value is stored. When we use string `'apple'` key, which is hashable in Python, the hash value is what gives us a way each time to get its value back with **no scanning the entire data structure**, but getting it instantly by computing the hash key every time and pulling out the value.

Right idea, but let's get concrete on the mechanism. In Python, `hash("apple")` might return something like `-4522842391661652542` — a huge number, positive or negative. But the underlying storage backing a hash map is a small fixed-size array, say with 8 slots.

**How does a hash value like that turn into "put this in slot 3"?**
- It's simple. The given hash value is mod (`%`) by the size of the array (say 8 slots). This gives a clear slot to store the value. Hash values could be the same for different objects that lead to the known hash map problem - hash collision. There are a bunch of solutions to solve it, linear probing, linked lists, etc. Then, once a certain threshold is hit, say 0.75, the hash map gets resized by doubling the array size and reindexing the whole array, which indeed takes $O(n)$. Python doubles the array size giving extra slots for spare, that makes resizing not that often happening and in generaly amortizes insertion operation to $O(1)$ eventually.

Solid:
- **mod** for slot placement
- **chaining/probing** for collisions
- **load factor triggering** amortized resizing.
That's the real mechanism, not just the marketing pitch of "$O(1)$."

**One more piece before we call the concept done, tying back to Section 4's rule about best/average/worst case: you've described _why_ lookup is $O(1)$ on average. Given that collisions exist and get resolved via probing or chaining, what's the **worst-case** time complexity for a single hash map lookup — and what has to go wrong for that worst case to actually happen?**
- It generates a probe sequence - a sequence of valid indices inside the hash map. It iterates over the probe sequence by index looking for an empty slot. If this slot is empty it stores the new item there (hash, key, value), if not - hash collision. It continues iteration to look for an empty slot. So it repeats looping until an empty slot is found to store the item. Iteration costs $O(n)$ - this is the worst case for hash map, the more hash collisions the more inefficient the hash map becomes having to go through this probe sequence quite a bit before we finally found an empty slot.

Right — worst case $O(n)$, because in the degenerate scenario the probe sequence (or chain) has to walk through almost every slot before finding what it's looking for.

**But be precise about "the more collisions, the worse it gets" — what's the specific pathological case where a hash map degrades to _exactly_ $O(n)$ for every single lookup, not just "somewhat slower"? Think about what would have to be true of the hash function's output across all your keys.**
- Thinking about the hash function's output, it is possible that it might be not that good (efficient) in producing distinct output for many different objects. It might really give hash values that repeat more often if this hash function is not efficient.

**Right direction — push it to the extreme. What if it's not just "repeats more often" but _every single key_ you insert happens to hash to the **same** slot? Structurally, what does the hash map become at that point? What data structure does it start behaving like?**
- If the hash function produces the same hash value the hash map becomes to behave like an array, scanning on each operation the whole structure.

Exactly.

> **Worst case**: every key collides into **one slot**, and the hash map degenerates into a linear scan — $O(n)$ per lookup/insert/delete, same as searching an unsorted array. That's the whole "average $O(1)$" claim depending entirely on the hash function distributing keys well.

Real-world note since you've now got the full mechanism: this isn't just theoretical. It's called **hash flooding** — an attacker who knows your hash function can craft keys that all collide, forcing every operation to $O(n)$ and taking your service down. It's why Python randomizes the hash seed for strings by default (`PYTHONHASHSEED`), so an attacker can't predict collisions across runs.
```shell
Average: O(1) — hash function distributes keys evenly across slots
Worst:   O(n) — all keys collide into one slot/chain, degenerates to linear scan
Space:   O(n) — one entry stored per key
```
You've got the concept solid — mechanism, collisions, resizing, and now the failure mode. Let's put it to work.